# LeetCode 1918 — Kth Smallest Subarray Sum bằng Python

Link đề: https://leetcode.com/problems/kth-smallest-subarray-sum/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Sliding Window

> 🔒 **Bài Premium.** Đề trong notebook được viết lại theo hiểu biết về bài (nguồn: kho lời giải doocs/leetcode). Nếu bạn có tài khoản Premium, nên đối chiếu lại câu chữ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def kthSmallestSubarraySum(self, nums: List[int], k: int) -> int:
        ...
```

Lộ trình: **liệt kê mọi tổng mảng con rồi sắp xếp** `O(n² log n)` → **heap trộn `n` dãy tăng** `O((n + k) log n)` → **tìm kiếm nhị phân trên đáp án + đếm bằng tổng tiền tố & `bisect`** `O(n log n · log S)` → **tìm kiếm nhị phân + cửa sổ trượt đếm** `O(n log S)` (lời giải nộp).

## 1. Mục tiêu học

1. Nhận ra mẫu **"phần tử nhỏ thứ k trong một tập ẩn rất lớn"** ⇒ **tìm kiếm nhị phân trên giá trị** + hàm **đếm số phần tử ≤ x**.
2. Dùng **cửa sổ trượt** (hai con trỏ) để đếm số mảng con có tổng ≤ `x` trong `O(n)` — nhờ mọi phần tử **dương**.
3. Viết đúng mẫu nhị phân "**giá trị nhỏ nhất** `x` sao cho `count(x) ≥ k`".
4. Biết cách **trộn nhiều dãy tăng bằng heap** — tốt khi `k` nhỏ.
5. Kiểm chứng bằng liệt kê-sắp xếp trên input ngẫu nhiên có seed; hiểu vì sao không được bỏ tổng trùng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n` và số nguyên `k`, trả về **tổng mảng con nhỏ thứ `k`** (mảng con là đoạn liên tiếp khác rỗng; tổng mảng con là tổng các phần tử của nó). Các tổng **trùng nhau vẫn tính riêng**.

**Ví dụ 1:** `nums = [2,1,3], k = 4` → `3`. Các tổng: `2, 1, 3, 3 ([2,1]), 4 ([1,3]), 6` ⇒ sắp xếp `1, 2, 3, 3, 4, 6` ⇒ thứ 4 là `3`.

**Ví dụ 2:** `nums = [3,3,5,5], k = 7` → `10`. Sắp xếp các tổng: `3, 3, 5, 5, 6, 8, 10, 11, 13, 16` ⇒ thứ 7 là `10`.

**Ràng buộc:** `1 ≤ n ≤ 2·10⁴`, `1 ≤ nums[i] ≤ 5·10⁴`, `1 ≤ k ≤ n(n+1)/2`.

In [1]:
import heapq
import random
import time
from bisect import bisect_right
from itertools import accumulate
from typing import List

examples = [([2, 1, 3], 4, 3), ([3, 3, 5, 5], 7, 10)]
for nums, k, want in examples:
    print(nums, "k =", k, "-> kỳ vọng", want)

[2, 1, 3] k = 4 -> kỳ vọng 3
[3, 3, 5, 5] k = 7 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1 Tổng tiền tố và tổng đoạn

`P[0] = 0`, `P[j] = nums[0] + … + nums[j−1]`. Tổng `nums[i..j−1] = P[j] − P[i]`. Vì mọi `nums[i] > 0`, `P` **tăng ngặt**.

In [2]:
nums = [3, 3, 5, 5]
P = list(accumulate(nums, initial=0))
print("P =", P, "| tổng nums[1..2] =", P[3] - P[1])

P = [0, 3, 6, 11, 16] | tổng nums[1..2] = 8


### 3.2 `bisect_right` trên mảng tăng

`bisect_right(P, v)` = số phần tử `≤ v`. Dùng để đếm: với mỗi `j`, số `i < j` có `P[j] − P[i] ≤ x` ⇔ `P[i] ≥ P[j] − x`.

In [3]:
P = [0, 3, 6, 11, 16]
print("số phần tử ≤ 6:", bisect_right(P, 6), "| ≤ 5:", bisect_right(P, 5))

số phần tử ≤ 6: 3 | ≤ 5: 2


### 3.3 Cửa sổ trượt trên mảng dương

Với số **dương**, mở rộng cửa sổ thì tổng **tăng**, thu hẹp thì **giảm** ⇒ với mỗi đầu phải `r`, đầu trái nhỏ nhất `l` sao cho tổng `nums[l..r] ≤ x` chỉ **tiến**, không lùi. Số mảng con kết thúc tại `r` có tổng `≤ x` là `r − l + 1`.

In [4]:
def count_le_window(nums, x):
    cnt = s = l = 0
    for r, v in enumerate(nums):
        s += v
        while s > x:
            s -= nums[l]; l += 1
        cnt += r - l + 1
    return cnt

print([count_le_window([3, 3, 5, 5], x) for x in [2, 3, 5, 6, 10, 16]])

[0, 2, 4, 5, 7, 10]


### 3.4 Mẫu nhị phân "nhỏ nhất thỏa điều kiện"

`count(x)` **không giảm** theo `x`. Tìm `x` nhỏ nhất với `count(x) ≥ k`:

```python
lo, hi = thấp, cao                 # count(hi) ≥ k chắc chắn
while lo < hi:
    mid = (lo + hi) // 2           # làm tròn XUỐNG vì nhánh thành công là hi = mid
    if count(mid) >= k: hi = mid
    else:               lo = mid + 1
```

In [5]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid): hi = mid
        else: lo = mid + 1
    return lo

print("x nhỏ nhất với x*x >= 50:", first_true(0, 50, lambda x: x * x >= 50))

x nhỏ nhất với x*x >= 50: 8


### 3.5 Heap trộn nhiều dãy đã sắp xếp

Có `n` dãy tăng; heap giữ phần tử **đầu** của mỗi dãy. Lấy nhỏ nhất ra rồi đẩy phần tử kế tiếp của **cùng dãy** — lấy `k` lần cho ra `k` phần tử nhỏ nhất của hợp các dãy.

In [6]:
lists = [[1, 4, 9], [2, 3, 10], [5]]
h = [(lst[0], i, 0) for i, lst in enumerate(lists)]
heapq.heapify(h)
out = []
while h:
    v, i, j = heapq.heappop(h); out.append(v)
    if j + 1 < len(lists[i]):
        heapq.heappush(h, (lists[i][j + 1], i, j + 1))
print(out)

[1, 2, 3, 4, 5, 9, 10]


### 3.6 Đếm cặp `(i, j)` thay vì liệt kê

Câu hỏi "có bao nhiêu cặp `i < j` với `P[j] − P[i] ≤ x`" trả lời được **không cần** liệt kê các cặp: cố định `j`, các `i` hợp lệ tạo thành một **khối liên tiếp** (vì `P` tăng) — chỉ cần biết biên của khối. Đây là ý tưởng chung đằng sau cả cách `bisect` (tìm biên bằng nhị phân) và cách cửa sổ trượt (biên chỉ tiến).

In [7]:
P = [0, 3, 6, 11, 16]; x = 10
for j in range(1, len(P)):
    ok = [i for i in range(j) if P[j] - P[i] <= x]
    print(f"j={j}: các i hợp lệ {ok} (một khối liên tiếp, bắt đầu ở {ok[0] if ok else '-'})")

j=1: các i hợp lệ [0] (một khối liên tiếp, bắt đầu ở 0)
j=2: các i hợp lệ [0, 1] (một khối liên tiếp, bắt đầu ở 0)
j=3: các i hợp lệ [1, 2] (một khối liên tiếp, bắt đầu ở 1)
j=4: các i hợp lệ [2, 3] (một khối liên tiếp, bắt đầu ở 2)


## 4. Ý tưởng nền tảng / trực giác

Có tới `n(n+1)/2 ≈ 2·10⁸` mảng con — không thể liệt kê. Đổi câu hỏi: "**có bao nhiêu** tổng mảng con `≤ x`?" — gọi `count(x)`. Ví dụ 2 `[3,3,5,5]`:

| x | các tổng ≤ x | `count(x)` |
|---|---|---|
| 5 | 3, 3, 5, 5 | 4 |
| 6 | + 6 | 5 |
| 8 | + 8 | 6 |
| 9 | (không thêm) | 6 |
| **10** | + 10 | **7** |
| 11 | + 11 | 8 |

Tổng nhỏ thứ `k = 7` là `x` **nhỏ nhất** có `count(x) ≥ 7` ⇒ `10`. ✔ Và `count(x)` tính được nhanh nhờ mảng toàn số dương.

In [8]:
for x in [5, 6, 8, 9, 10, 11]:
    print(x, count_le_window([3, 3, 5, 5], x))

5 4
6 5
8 6
9 6
10 7
11 8


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi tổng rồi sắp xếp

In [9]:
def kth_brute(nums, k):
    sums = []
    for i in range(len(nums)):
        s = 0
        for j in range(i, len(nums)):
            s += nums[j]
            sums.append(s)
    sums.sort()
    return sums[k - 1]

for nums, k, want in examples:
    print(kth_brute(nums, k), "| kỳ vọng", want)

3 | kỳ vọng 3
10 | kỳ vọng 10


**Độ phức tạp:** `O(n²)` tổng + sắp xếp `O(n² log n)`; bộ nhớ `O(n²)` — với `n = 2·10⁴` là `2·10⁸` số (vài GB). **Nút thắt:** ta chỉ cần **một** phần tử của danh sách đã sắp xếp, không cần cả danh sách.

In [10]:
rng = random.Random(0)
arr = [rng.randint(1, 5 * 10**4) for _ in range(1500)]
t = time.perf_counter(); kth_brute(arr, 12345); dt = time.perf_counter() - t
print(f"n=1500 (~1.1·10^6 tổng): {dt:.2f}s -> n=2·10^4 ước tính ≈ {dt * (2e4 / 1500) ** 2:.0f}s và ~1.6 GB bộ nhớ")

n=1500 (~1.1·10^6 tổng): 0.17s -> n=2·10^4 ước tính ≈ 30s và ~1.6 GB bộ nhớ


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (nhị phân trên đáp án):** `count(x)` = số mảng con có tổng `≤ x` là hàm **không giảm**. Tổng nhỏ thứ `k` là `x* = min{x : count(x) ≥ k}`. *Vì sao `x*` là một tổng thật?* Tại `x* − 1` có `count < k`, tại `x*` có `count ≥ k` ⇒ có mảng con tổng **đúng bằng** `x*`.

**Insight 2 (đếm nhanh):** mọi phần tử dương ⇒ cửa sổ trượt đếm `count(x)` trong `O(n)`.

**Khoảng tìm kiếm:** `lo = min(nums)` (tổng nhỏ nhất), `hi = sum(nums)` (tổng lớn nhất, `count = n(n+1)/2 ≥ k`). Số vòng `log₂(10⁹) ≈ 30`.

**Insight 3 (heap, khi `k` nhỏ):** với mỗi điểm đầu `i`, dãy `P[i+1]−P[i], P[i+2]−P[i], …` **tăng** ⇒ trộn `n` dãy bằng heap, lấy `k` lần.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Heap trộn `n` dãy tăng (`O((n + k) log n)`)

In [11]:
def kth_heap(nums, k):
    n = len(nums)
    P = list(accumulate(nums, initial=0))
    h = [(nums[i], i, i) for i in range(n)]          # (tổng nums[i..j], i, j)
    heapq.heapify(h)
    for _ in range(k - 1):
        s, i, j = heapq.heappop(h)
        if j + 1 < n:
            heapq.heappush(h, (P[j + 2] - P[i], i, j + 1))
    return h[0][0]

for nums, k, want in examples:
    print(kth_heap(nums, k), want)

3 3
10 10


**Độ phức tạp:** `O(n + k log n)`. Rất tốt khi `k` nhỏ, nhưng `k` tới `2·10⁸` ⇒ không dùng được worst-case.

### 7.2 Cách 3 — Nhị phân + đếm bằng tiền tố & `bisect` (`O(n log n · log S)`)

In [12]:
def count_le_bisect(P, x):
    # với mỗi j: số i < j có P[i] >= P[j] - x  =  j - (số i < j có P[i] < P[j] - x)
    total = 0
    for j in range(1, len(P)):
        total += j - bisect_right(P, P[j] - x - 1, 0, j)
    return total

class SolutionBisect:
    def kthSmallestSubarraySum(self, nums: List[int], k: int) -> int:
        P = list(accumulate(nums, initial=0))
        lo, hi = min(nums), P[-1]
        while lo < hi:
            mid = (lo + hi) // 2
            if count_le_bisect(P, mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

for nums, k, want in examples:
    print(SolutionBisect().kthSmallestSubarraySum(nums, k), want)

3 3
10 10


`bisect_right(P, v, 0, j)` đếm số `P[i] ≤ v` với `i < j`; đặt `v = P[j] − x − 1` (số nguyên) để đếm `P[i] < P[j] − x`. **Độ phức tạp:** mỗi `count` `O(n log n)`, ~30 vòng ⇒ `O(n log n log S)`.

### 7.3 Cách 4 — Nhị phân + cửa sổ trượt (`O(n log S)`) — lời giải chuẩn

In [13]:
def kth_window_demo(nums, k):
    lo, hi = min(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        c = count_le_window(nums, mid)
        print(f"lo={lo:>3} hi={hi:>3} mid={mid:>3} count(mid)={c:>3} {'>= k -> hi=mid' if c >= k else '< k -> lo=mid+1'}")
        if c >= k: hi = mid
        else: lo = mid + 1
    return lo

print("=>", kth_window_demo([3, 3, 5, 5], 7))

lo=  3 hi= 16 mid=  9 count(mid)=  6 < k -> lo=mid+1
lo= 10 hi= 16 mid= 13 count(mid)=  9 >= k -> hi=mid
lo= 10 hi= 13 mid= 11 count(mid)=  8 >= k -> hi=mid
lo= 10 hi= 11 mid= 10 count(mid)=  7 >= k -> hi=mid
=> 10


In [14]:
class SolutionWindow:
    def kthSmallestSubarraySum(self, nums: List[int], k: int) -> int:
        def count(x):
            c = s = l = 0
            for r, v in enumerate(nums):
                s += v
                while s > x:
                    s -= nums[l]; l += 1
                c += r - l + 1
            return c
        lo, hi = min(nums), sum(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if count(mid) >= k: hi = mid
            else: lo = mid + 1
        return lo

for nums, k, want in examples:
    print(SolutionWindow().kthSmallestSubarraySum(nums, k), want)

3 3
10 10


**Độ phức tạp:** mỗi `count` `O(n)` (mỗi con trỏ đi tối đa `n` bước), ~30 vòng ⇒ `O(n log S)` với `S = sum(nums) ≤ 10⁹`. Bộ nhớ `O(1)`.

## 8. TRACE từng bước

Trace hàm đếm cửa sổ trượt cho `nums = [3,3,5,5]`, `x = 10`:

| r | v | `s` sau cộng | thu hẹp | `l` | đoạn hợp lệ kết thúc tại r | cộng |
|---|---|---|---|---|---|---|
| 0 | 3 | 3 | — | 0 | `[3]` | 1 |
| 1 | 3 | 6 | — | 0 | `[3]`, `[3,3]` | 2 |
| 2 | 5 | 11 | bỏ 3 → 8 | 1 | `[5]`, `[3,5]` | 2 |
| 3 | 5 | 13 | bỏ 3 → 10 | 2 | `[5]`, `[5,5]` | 2 |

Tổng `count(10) = 7 ≥ 7`; còn `count(9) = 6 < 7` ⇒ đáp án 10. ✔

In [15]:
def trace_count(nums, x):
    c = s = l = 0
    for r, v in enumerate(nums):
        s += v
        removed = []
        while s > x:
            removed.append(nums[l]); s -= nums[l]; l += 1
        c += r - l + 1
        print(f"r={r} v={v}: bỏ {removed} -> s={s}, l={l}, đoạn [{l}..{r}] .. [{r}..{r}] => +{r - l + 1}, count={c}")
    return c

trace_count([3, 3, 5, 5], 10); print(); trace_count([3, 3, 5, 5], 9)

r=0 v=3: bỏ [] -> s=3, l=0, đoạn [0..0] .. [0..0] => +1, count=1
r=1 v=3: bỏ [] -> s=6, l=0, đoạn [0..1] .. [1..1] => +2, count=3
r=2 v=5: bỏ [3] -> s=8, l=1, đoạn [1..2] .. [2..2] => +2, count=5
r=3 v=5: bỏ [3] -> s=10, l=2, đoạn [2..3] .. [3..3] => +2, count=7

r=0 v=3: bỏ [] -> s=3, l=0, đoạn [0..0] .. [0..0] => +1, count=1
r=1 v=3: bỏ [] -> s=6, l=0, đoạn [0..1] .. [1..1] => +2, count=3
r=2 v=5: bỏ [3] -> s=8, l=1, đoạn [1..2] .. [2..2] => +2, count=5
r=3 v=5: bỏ [3, 5] -> s=5, l=3, đoạn [3..3] .. [3..3] => +1, count=6


6

Trace heap trên Ví dụ 1 (`k = 4`): heap ban đầu `(2,0,0), (1,1,1), (3,2,2)`.

In [16]:
def trace_heap(nums, k):
    n = len(nums); P = list(accumulate(nums, initial=0))
    h = [(nums[i], i, i) for i in range(n)]; heapq.heapify(h)
    for step in range(1, k):
        s, i, j = heapq.heappop(h)
        print(f"lần {step}: lấy tổng {s} = nums[{i}..{j}]", end="")
        if j + 1 < n:
            heapq.heappush(h, (P[j + 2] - P[i], i, j + 1)); print(f", đẩy nums[{i}..{j+1}] = {P[j+2]-P[i]}")
        else:
            print()
    print("thứ", k, "là", h[0][0])
    return h[0][0]

trace_heap([2, 1, 3], 4)

lần 1: lấy tổng 1 = nums[1..1], đẩy nums[1..2] = 4
lần 2: lấy tổng 2 = nums[0..0], đẩy nums[0..1] = 3
lần 3: lấy tổng 3 = nums[0..1], đẩy nums[0..2] = 6
thứ 4 là 3


3

## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def kthSmallestSubarraySum(self, nums: List[int], k: int) -> int:
        def count(x: int) -> int:            # số mảng con có tổng <= x (mọi phần tử dương)
            c = s = l = 0
            for r, v in enumerate(nums):
                s += v
                while s > x:
                    s -= nums[l]
                    l += 1
                c += r - l + 1
            return c

        lo, hi = min(nums), sum(nums)
        while lo < hi:                       # x nhỏ nhất có count(x) >= k
            mid = (lo + hi) // 2
            if count(mid) >= k:
                hi = mid
            else:
                lo = mid + 1
        return lo

for nums, k, want in examples:
    assert Solution().kthSmallestSubarraySum(nums, k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Hàm đếm.** Cố định đầu phải `r`. Vì phần tử dương, `sum(nums[l..r])` **giảm** khi `l` tăng; tập `l` hợp lệ (tổng `≤ x`) là một đoạn `[l*(r), r]` (có thể rỗng nếu `nums[r] > x`, khi đó vòng `while` đẩy `l = r + 1` và cộng `0`). Khi `r` tăng, `l*(r)` không giảm (thêm phần tử chỉ làm tổng lớn hơn) ⇒ con trỏ `l` chỉ tiến ⇒ vòng `while` tính đúng `l*(r)`. Cộng `r − l*(r) + 1` cho mọi `r` = `count(x)`.

**Nhị phân.** Gọi `s₁ ≤ s₂ ≤ … ≤ s_N` là các tổng (có lặp). `count(x) ≥ k ⇔ s_k ≤ x`. Vậy `min{x : count(x) ≥ k} = s_k`. Khoảng ban đầu `[min(nums), sum(nums)]` chứa `s_k` (vì `s₁ = min(nums)`, `s_N = sum(nums)`). Mẫu nhị phân giữ bất biến "`count(hi) ≥ k`, mọi `x < lo` có `count(x) < k`" và dừng khi `lo = hi = s_k`.

**Heap.** Mỗi dãy (theo điểm đầu `i`) tăng ngặt; heap luôn chứa phần tử nhỏ nhất chưa lấy của mỗi dãy ⇒ phần tử pop thứ `t` là `s_t` (chứng minh chuẩn của k-way merge).

Kiểm chứng hai mệnh đề then chốt bằng máy trên nhiều mảng ngẫu nhiên: (1) `count(x)` **không giảm** theo `x`; (2) đáp án nhị phân **luôn là một tổng mảng con thật** — dù ta không hề kiểm điều đó trong code.

In [18]:
rng_c = random.Random(101)
for _ in range(200):
    nums = [rng_c.randint(1, 9) for _ in range(rng_c.randint(1, 10))]
    vals = [count_le_window(nums, x) for x in range(0, sum(nums) + 2)]
    assert all(vals[i] <= vals[i + 1] for i in range(len(vals) - 1))         # (1)
    sums = {sum(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1)}
    N = len(nums) * (len(nums) + 1) // 2
    for k in range(1, N + 1):
        assert Solution().kthSmallestSubarraySum(nums, k) in sums             # (2)
print("count(x) không giảm; đáp án nhị phân luôn thuộc tập tổng thật")

count(x) không giảm; đáp án nhị phân luôn thuộc tập tổng thật


Trực giác cho (2): nếu `x*` không phải tổng nào, thì `count(x* − 1) = count(x*) ≥ k`, mâu thuẫn với việc `x*` là giá trị **nhỏ nhất** có `count ≥ k`.

## 11. Phân tích Time / Space

Gọi `n ≤ 2·10⁴`, `S = sum(nums) ≤ 10⁹`, `N = n(n+1)/2`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê + sắp xếp | `O(N log N)` | `O(N)` |
| 2. Heap trộn | `O(n + k log n)` | `O(n)` |
| 3. Nhị phân + bisect | `O(n log n · log S)` | `O(n)` |
| 4. Nhị phân + cửa sổ | `O(n · log S)` | `O(1)` |

In [19]:
rng = random.Random(1918)
big = [rng.randint(1, 5 * 10**4) for _ in range(2 * 10**4)]
N = len(big) * (len(big) + 1) // 2
for k in [1, N // 2, N]:
    res = []
    for name, sol in [("bisect", SolutionBisect()), ("cửa sổ/Nộp", Solution())]:
        t = time.perf_counter(); r = sol.kthSmallestSubarraySum(big, k); res.append(r)
        print(f"k={k}: {name}: {r} trong {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1
t = time.perf_counter(); r = kth_heap(big, 5000); dt = time.perf_counter() - t
print(f"heap với k=5000: {r} trong {1000*dt:.0f} ms (khớp: {r == Solution().kthSmallestSubarraySum(big, 5000)})")

k=1: bisect: 5 trong 74 ms
k=1: cửa sổ/Nộp: 5 trong 49 ms


k=100005000: bisect: 146634854 trong 83 ms
k=100005000: cửa sổ/Nộp: 146634854 trong 54 ms


k=200010000: bisect: 499782118 trong 70 ms
k=200010000: cửa sổ/Nộp: 499782118 trong 34 ms


heap với k=5000: 11058 trong 8 ms (khớp: True)


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu liệt kê)

In [20]:
sols = [Solution(), SolutionWindow(), SolutionBisect()]
edge = examples + [
    ([5], 1, 5),
    ([1, 1, 1], 4, 2),               # tổng: 1,1,1,2,2,3
    ([1, 1, 1], 6, 3),
    ([1, 2, 3], 1, 1),
    ([4, 1], 3, 5),                  # tổng: 1,4,5
    ([50000] * 5, 15, 250000),       # k = N: tổng cả mảng
]
for nums, k, want in edge:
    assert kth_brute(nums, k) == want
    for s in sols:
        assert s.kthSmallestSubarraySum(nums, k) == want, (nums, k, s)
    assert kth_heap(nums, k) == want
    print("OK", nums, "k =", k, "->", want)

OK [2, 1, 3] k = 4 -> 3
OK [3, 3, 5, 5] k = 7 -> 10
OK [5] k = 1 -> 5
OK [1, 1, 1] k = 4 -> 2
OK [1, 1, 1] k = 6 -> 3
OK [1, 2, 3] k = 1 -> 1
OK [4, 1] k = 3 -> 5
OK [50000, 50000, 50000, 50000, 50000] k = 15 -> 250000


### 12.2 Ngẫu nhiên có seed: mọi `k` trên mảng nhỏ

In [21]:
rng = random.Random(42)
for _ in range(200):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, rng.choice([3, 20, 5 * 10**4])) for _ in range(n)]
    N = n * (n + 1) // 2
    sums = sorted(sum(nums[i:j]) for i in range(n) for j in range(i + 1, n + 1))
    for k in range(1, N + 1):
        assert Solution().kthSmallestSubarraySum(nums, k) == sums[k - 1]
        assert SolutionBisect().kthSmallestSubarraySum(nums, k) == sums[k - 1]
        assert kth_heap(nums, k) == sums[k - 1]
print("200 mảng × mọi k: khớp danh sách sắp xếp")

200 mảng × mọi k: khớp danh sách sắp xếp


### 12.3 Hàm đếm khớp định nghĩa

In [22]:
rng = random.Random(7)
for _ in range(300):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 15))]
    P = list(accumulate(nums, initial=0))
    x = rng.randint(0, sum(nums) + 5)
    direct = sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if P[j] - P[i] <= x)
    assert count_le_window(nums, x) == direct == count_le_bisect(P, x)
print("count_le (cửa sổ và bisect) khớp định nghĩa")

count_le (cửa sổ và bisect) khớp định nghĩa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Khử trùng các tổng (`set`)

In [23]:
def wrong_set(nums, k):
    sums = sorted({sum(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1)})   # SAI
    return sums[k - 1] if k <= len(sums) else None

for nums, k, want in examples:
    print(nums, k, "sai:", wrong_set(nums, k), "| đúng:", want)

[2, 1, 3] 4 sai: 4 | đúng: 3
[3, 3, 5, 5] 7 sai: 13 | đúng: 10


**Vì sao sai:** đề đếm **mọi** mảng con, tổng trùng nhau vẫn chiếm chỗ riêng (Ví dụ 1 có hai tổng `3`). Bỏ trùng thì thứ hạng lệch (thậm chí `k` vượt số phần tử).

### Lỗi 2 — Nhị phân dùng `count(mid) > k` hoặc trả `mid`

In [24]:
def wrong_bs(nums, k):
    lo, hi = min(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le_window(nums, mid) > k:      # SAI: phải là >= k
            hi = mid
        else:
            lo = mid + 1
    return lo

for nums, k, want in examples + [([1, 1, 1], 4, 2)]:
    print(nums, k, "sai:", wrong_bs(nums, k), "| đúng:", want)

[2, 1, 3] 4 sai: 4 | đúng: 3
[3, 3, 5, 5] 7 sai: 11 | đúng: 10
[1, 1, 1] 4 sai: 2 | đúng: 2


**Vì sao sai:** ta cần `x` nhỏ nhất mà **ít nhất** `k` tổng `≤ x`. Dùng `> k` tìm `x` nhỏ nhất có **hơn** `k` tổng — trỏ sang phần tử thứ `k + 1` (hoặc tổng lớn hơn khi có trùng).

### Lỗi 3 — Đếm cửa sổ sai: cộng `1` thay vì `r − l + 1`

In [25]:
def wrong_count(nums, x):
    c = s = l = 0
    for r, v in enumerate(nums):
        s += v
        while s > x:
            s -= nums[l]; l += 1
        c += 1 if l <= r else 0                 # SAI: chỉ đếm đoạn dài nhất
    return c

print("sai:", [wrong_count([3, 3, 5, 5], x) for x in (6, 10)], "| đúng:", [count_le_window([3, 3, 5, 5], x) for x in (6, 10)])

sai: [4, 4] | đúng: [5, 7]


**Vì sao sai:** với đầu phải `r` cố định, **mọi** đầu trái `l..r` đều cho tổng `≤ x` (vì bỏ bớt phần tử dương làm tổng nhỏ hơn) ⇒ có `r − l + 1` mảng con, không phải 1.

### Lỗi 4 — Khoảng nhị phân bắt đầu từ `0`, cận trên `max(nums)`

In [26]:
def wrong_range(nums, k):
    lo, hi = 0, max(nums)                       # SAI: tổng có thể lớn hơn max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le_window(nums, mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

for nums, k, want in examples:
    print(nums, k, "sai:", wrong_range(nums, k), "| đúng:", want)

[2, 1, 3] 4 sai: 3 | đúng: 3
[3, 3, 5, 5] 7 sai: 5 | đúng: 10


**Vì sao sai:** đáp án có thể tới `sum(nums)`; cận trên `max(nums)` cắt mất phần lớn miền ⇒ trả về `max(nums)` khi đáp án thật lớn hơn. (Cận dưới `0` thì không sai, chỉ tốn vài vòng.) **Sửa:** `hi = sum(nums)`.

### Lỗi 5 — Heap đẩy sẵn mọi tổng (`O(n²)` bộ nhớ)

In [27]:
def heap_all(nums, k):
    h = [sum(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1)]
    heapq.heapify(h)
    return [heapq.heappop(h) for _ in range(k)][-1]

arr = [rng.randint(1, 100) for _ in range(400)]
t = time.perf_counter(); r1 = heap_all(arr, 1000); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = kth_heap(arr, 1000); t2 = time.perf_counter() - t
print(f"đẩy hết: {1000*t1:.0f} ms | trộn n dãy: {1000*t2:.1f} ms | cùng kết quả: {r1 == r2}")

đẩy hết: 60 ms | trộn n dãy: 0.4 ms | cùng kết quả: True


**Vì sao chưa tốt:** đúng nhưng `O(n²)` bộ nhớ/thời gian (lại thêm `sum` lát cắt `O(n)` mỗi tổng). Heap chỉ cần giữ **một** ứng viên cho mỗi điểm đầu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Phần tử nhỏ thứ k trong một tập ẩn khổng lồ"** (tổng mảng con, khoảng cách cặp, phần tử ma trận đã sắp xếp…) ⇒ **nhị phân trên giá trị** + `count(x)`.
- Mảng **toàn số dương** ⇒ đếm mảng con có tổng `≤ x` bằng **cửa sổ trượt** `O(n)`.
- Mẫu "nhỏ nhất thỏa": `mid = (lo + hi) // 2`, thành công `hi = mid`, thất bại `lo = mid + 1`.
- `k` nhỏ ⇒ heap trộn các dãy tăng.
- Có số âm ⇒ cửa sổ trượt **không** dùng được; đếm bằng tiền tố + cấu trúc có thứ tự (BIT / merge sort).

## 15. Biến thể của bài

1. **LeetCode 719 — K-th Smallest Pair Distance:** nhị phân + hai con trỏ trên mảng đã sắp xếp (Bài tập 5).
2. **LeetCode 378 — K-th Smallest Element in a Sorted Matrix:** nhị phân + đếm theo hàng/cột, hoặc heap.
3. **Tổng lớn thứ `k`** ⇒ tổng nhỏ thứ `N − k + 1` (Bài tập 4).
4. **Mảng có số âm** ⇒ đếm `P[j] − P[i] ≤ x` bằng merge sort / BIT trên tiền tố đã nén.
5. **LeetCode 1508 — Range Sum of Sorted Subarray Sums:** tổng các phần tử thứ `left..right` — nhị phân + đếm kèm tổng.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê + sắp xếp | trực tiếp | `O(n² log n)` | `O(n²)` | `n` ≤ vài nghìn, kiểm chứng |
| 2. Heap trộn | `n` dãy tăng | `O(n + k log n)` | `O(n)` | `k` nhỏ |
| 3. Nhị phân + bisect | đếm bằng tiền tố | `O(n log n log S)` | `O(n)` | quen bisect |
| 4. Nhị phân + cửa sổ | đếm `O(n)` | `O(n log S)` | `O(1)` | **nộp bài** |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `([1,1,1], 4)`, `([1,2,3], 6)`, `([10], 1)`?

**Bài 2 (tính tay):** Ghi các vòng nhị phân (`lo`, `hi`, `mid`, `count(mid)`) cho Ví dụ 1 `([2,1,3], 4)`.

**Bài 3 (điền TODO):** Viết `count_lt(nums, x)` — số mảng con có tổng **nhỏ hơn ngặt** `x`, bằng cửa sổ trượt.

**Bài 4 (tự viết lại):** Viết `kth_largest_subarray_sum(nums, k)` bằng nhị phân trực tiếp (đếm số tổng `≥ x`), không quy về bài gốc.

**Bài 5 (mini challenge — kiểu LeetCode 719):** Cho mảng số nguyên không âm `a` và `k`, trả về **khoảng cách nhỏ thứ k** trong mọi cặp `|a[i] − a[j]|` (`i < j`).

In [28]:
def count_lt(nums, x):
    # TODO
    pass

def kth_largest_subarray_sum(nums, k):
    # TODO
    pass

def kth_smallest_distance(a, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Liệt kê tổng | `[1,1,1]`: 1,1,1,2,2,3 | `[1,2,3]` thứ 6 = 6; `[10]` = 10 |
| 2 | `lo = 1, hi = 6` | `mid = 3`: count = 4 ≥ 4 | thu hẹp tới `lo = hi = 3` |
| 3 | `count_lt(x) = count_le(x − 1)` (số nguyên) | Hoặc đổi điều kiện `while s >= x` | Kiểm với định nghĩa |
| 4 | Số tổng `≥ x` = `N − count_le(x − 1)` | `x` **lớn nhất** có số đó `≥ k` | `mid = (lo + hi + 1) // 2` |
| 5 | Sắp xếp `a` | Đếm cặp có hiệu `≤ d` bằng hai con trỏ | Nhị phân `d ∈ [0, max − min]` |

## 19. Đáp án tham khảo

In [29]:
r1 = [Solution().kthSmallestSubarraySum(*a) for a in [([1, 1, 1], 4), ([1, 2, 3], 6), ([10], 1)]]
print("Bài 1:", r1); assert r1 == [2, 6, 10]
print("Bài 2:"); assert kth_window_demo([2, 1, 3], 4) == 3

Bài 1: [2, 6, 10]
Bài 2:
lo=  1 hi=  6 mid=  3 count(mid)=  4 >= k -> hi=mid
lo=  1 hi=  3 mid=  2 count(mid)=  2 < k -> lo=mid+1


In [30]:
def count_lt(nums, x):
    c = s = l = 0
    for r, v in enumerate(nums):
        s += v
        while l <= r and s >= x:
            s -= nums[l]; l += 1
        c += r - l + 1
    return c

def kth_largest_subarray_sum(nums, k):
    n = len(nums); N = n * (n + 1) // 2
    lo, hi = min(nums), sum(nums)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ge = N - count_le_window(nums, mid - 1)         # số tổng >= mid
        if ge >= k: lo = mid
        else: hi = mid - 1
    return lo

rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 10))]
    n = len(nums)
    sums = sorted(sum(nums[i:j]) for i in range(n) for j in range(i + 1, n + 1))
    x = rng.randint(0, sum(nums) + 3)
    assert count_lt(nums, x) == sum(1 for v in sums if v < x)
    for k in range(1, len(sums) + 1):
        assert kth_largest_subarray_sum(nums, k) == sums[-k]
print("Bài 3, 4: OK", kth_largest_subarray_sum([3, 3, 5, 5], 1), kth_largest_subarray_sum([3, 3, 5, 5], 4))

Bài 3, 4: OK 16 10


In [31]:
def kth_smallest_distance(a, k):
    a = sorted(a)
    def count(d):
        c = l = 0
        for r in range(len(a)):
            while a[r] - a[l] > d:
                l += 1
            c += r - l
        return c
    lo, hi = 0, a[-1] - a[0]
    while lo < hi:
        mid = (lo + hi) // 2
        if count(mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

assert kth_smallest_distance([1, 3, 1], 1) == 0          # ví dụ của LeetCode 719
assert kth_smallest_distance([1, 1, 1], 2) == 0
assert kth_smallest_distance([1, 6, 1], 3) == 5
rng = random.Random(9)
for _ in range(300):
    a = [rng.randint(0, 30) for _ in range(rng.randint(2, 12))]
    dists = sorted(abs(a[i] - a[j]) for i in range(len(a)) for j in range(i + 1, len(a)))
    k = rng.randint(1, len(dists))
    assert kth_smallest_distance(a, k) == dists[k - 1]
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def count(x):                         # số mảng con tổng <= x (phần tử dương)
    c = s = l = 0
    for r, v in enumerate(nums):
        s += v
        while s > x: s -= nums[l]; l += 1
        c += r - l + 1
    return c
lo, hi = min(nums), sum(nums)
while lo < hi:
    mid = (lo + hi) // 2
    if count(mid) >= k: hi = mid
    else: lo = mid + 1
return lo
```

**Checklist:**
- [ ] Đếm **mọi** mảng con (không bỏ tổng trùng)?
- [ ] Điều kiện nhị phân `count(mid) >= k`, nhánh thành công `hi = mid`?
- [ ] `hi = sum(nums)` (không phải `max(nums)`)?
- [ ] Cửa sổ cộng `r − l + 1` mỗi đầu phải?
- [ ] Ghi nhớ: cửa sổ trượt chỉ đúng vì mọi phần tử **dương**.